# Rule Disagreement Analysis

This notebook analyzes verifier rule trigger disagreements between gold UD pipeline output and Stanza pipeline output.

**Inputs:**
- `results/stanza_vs_gold_rule_disagreements.csv`
- `results/stanza_vs_gold_matched.csv`

**Purpose:** identify why R1 to R5 fire differently between gold UD and Stanza output.

Diagnostic only. No verifier, mapper, parser, or correction logic is modified here.

## 1. Load Data

In [1]:
import csv
from collections import Counter
from pathlib import Path

RESULTS_DIR = Path("../results")
RULE_DISAGREEMENTS_PATH = RESULTS_DIR / "stanza_vs_gold_rule_disagreements.csv"
MATCHED_PATH = RESULTS_DIR / "stanza_vs_gold_matched.csv"


def load_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


rule_disagreements = load_csv(RULE_DISAGREEMENTS_PATH)
matched_rows = load_csv(MATCHED_PATH)

print(f"Rule disagreement rows: {len(rule_disagreements)}")
print(f"Matched rows:           {len(matched_rows)}")

Rule disagreement rows: 206
Matched rows:           35217


## 2. Categorize Lost and Gained Rules

Definitions:
- Lost: gold fired a rule, Stanza did not.
- Gained: gold did not fire a rule, Stanza fired that rule.

In [2]:
RULES = ["R1", "R2", "R3", "R4", "R5"]


def category_rows(rule_id, direction):
    if direction == "lost":
        return [
            row for row in rule_disagreements
            if row["gold_verifier_rule_id"] == rule_id
            and row["stanza_verifier_rule_id"] == ""
        ]
    if direction == "gained":
        return [
            row for row in rule_disagreements
            if row["gold_verifier_rule_id"] == ""
            and row["stanza_verifier_rule_id"] == rule_id
        ]
    raise ValueError(f"Unknown direction: {direction}")


categories = {}
for rule_id in RULES:
    categories[f"{rule_id}_lost"] = category_rows(rule_id, "lost")
    categories[f"{rule_id}_gained"] = category_rows(rule_id, "gained")

total_rule_disagreements = len(rule_disagreements)
print(f"Total rule disagreements: {total_rule_disagreements}")
print()
for name, rows in categories.items():
    pct = 100 * len(rows) / total_rule_disagreements if total_rule_disagreements else 0
    print(f"{name:<12} {len(rows):>4} ({pct:>6.2f}%)")

Total rule disagreements: 206

R1_lost         4 (  1.94%)
R1_gained       8 (  3.88%)
R2_lost        16 (  7.77%)
R2_gained      33 ( 16.02%)
R3_lost         7 (  3.40%)
R3_gained      12 (  5.83%)
R4_lost        29 ( 14.08%)
R4_gained      48 ( 23.30%)
R5_lost        22 ( 10.68%)
R5_gained      27 ( 13.11%)


## 3. Cause Diagnosis Helpers

Cause labels are descriptive. They summarize the direct differences visible in the matched CSV columns.

In [3]:
EXAMPLE_FIELDS = [
    "sent_id",
    "sentence_text",
    "token_form",
    "gold_deprel",
    "stanza_deprel",
    "gold_case_marker",
    "stanza_case_marker",
    "gold_verifier_rule_id",
    "stanza_verifier_rule_id",
    "gold_final_decision",
    "stanza_final_decision",
]


def diagnose_cause(row):
    gold_deprel = row["gold_deprel"] or "empty"
    stanza_deprel = row["stanza_deprel"] or "empty"
    gold_case = row["gold_case_marker"] or "empty"
    stanza_case = row["stanza_case_marker"] or "empty"

    deprel_changed = gold_deprel != stanza_deprel
    case_changed = gold_case != stanza_case

    if deprel_changed and case_changed:
        return f"deprel changed {gold_deprel} to {stanza_deprel}; case marker changed {gold_case} to {stanza_case}"
    if deprel_changed:
        return f"deprel changed {gold_deprel} to {stanza_deprel}"
    if case_changed:
        return f"case marker changed {gold_case} to {stanza_case}"
    return "rule changed without visible deprel or case-marker difference"


def add_diagnosis(rows):
    diagnosed = []
    for row in rows:
        item = dict(row)
        item["diagnosis"] = diagnose_cause(row)
        diagnosed.append(item)
    return diagnosed


diagnosed_categories = {
    name: add_diagnosis(rows)
    for name, rows in categories.items()
}

for name, rows in diagnosed_categories.items():
    cause_counts = Counter(row["diagnosis"] for row in rows)
    most_common = cause_counts.most_common(1)[0] if cause_counts else ("none", 0)
    print(f"{name:<12} most common cause: {most_common[0]} ({most_common[1]})")

R1_lost      most common cause: case marker changed ने to empty (4)
R1_gained    most common cause: deprel changed conj to nsubj (4)
R2_lost      most common cause: deprel changed obl to nmod (5)
R2_gained    most common cause: deprel changed nmod to obl (19)
R3_lost      most common cause: case marker changed पर to empty (4)
R3_gained    most common cause: deprel changed nmod to obl (9)
R4_lost      most common cause: deprel changed obl to nmod (10)
R4_gained    most common cause: deprel changed obj to obl (25)
R5_lost      most common cause: deprel changed iobj to nsubj (12)
R5_gained    most common cause: deprel changed nsubj to obj (7)


## 4. Category Summary Table

In [4]:
summary_rows = []
for name, rows in diagnosed_categories.items():
    cause_counts = Counter(row["diagnosis"] for row in rows)
    most_common_cause = cause_counts.most_common(1)[0][0] if cause_counts else "none"
    pct = 100 * len(rows) / total_rule_disagreements if total_rule_disagreements else 0
    summary_rows.append({
        "category": name,
        "count": str(len(rows)),
        "percent_of_rule_disagreements": f"{pct:.2f}",
        "most_common_cause": most_common_cause,
    })

print(f"{'Category':<12} {'Count':>6} {'Percent':>10}  Most common cause")
print("=" * 90)
for row in summary_rows:
    print(
        f"{row['category']:<12} {row['count']:>6} "
        f"{row['percent_of_rule_disagreements']:>9}%  "
        f"{row['most_common_cause']}"
    )

Category      Count    Percent  Most common cause
R1_lost           4      1.94%  case marker changed ने to empty
R1_gained         8      3.88%  deprel changed conj to nsubj
R2_lost          16      7.77%  deprel changed obl to nmod
R2_gained        33     16.02%  deprel changed nmod to obl
R3_lost           7      3.40%  case marker changed पर to empty
R3_gained        12      5.83%  deprel changed nmod to obl
R4_lost          29     14.08%  deprel changed obl to nmod
R4_gained        48     23.30%  deprel changed obj to obl
R5_lost          22     10.68%  deprel changed iobj to nsubj
R5_gained        27     13.11%  deprel changed nsubj to obj


## 5. Display Category Examples

Each category prints up to 20 examples. If a category has fewer than 20 examples, all available rows are shown.

In [5]:
def show_category_examples(category_name, rows, max_examples=20):
    pct = 100 * len(rows) / total_rule_disagreements if total_rule_disagreements else 0
    print(f"{category_name}: {len(rows)} examples ({pct:.2f}% of rule disagreements)")
    print("=" * 90)
    if not rows:
        print("No examples in this category.")
        print()
        return
    for index, row in enumerate(rows[:max_examples], start=1):
        print(f"Example {index}")
        print(f"  sent_id: {row['sent_id']}")
        print(f"  sentence_text: {row['sentence_text']}")
        print(f"  token_form: {row['token_form']}")
        print(f"  gold_deprel: {row['gold_deprel']}")
        print(f"  stanza_deprel: {row['stanza_deprel']}")
        print(f"  gold_case_marker: {row['gold_case_marker']}")
        print(f"  stanza_case_marker: {row['stanza_case_marker']}")
        print(f"  gold_rule: {row['gold_verifier_rule_id']}")
        print(f"  stanza_rule: {row['stanza_verifier_rule_id']}")
        print(f"  gold_final_decision: {row['gold_final_decision']}")
        print(f"  stanza_final_decision: {row['stanza_final_decision']}")
        print(f"  diagnosis: {row['diagnosis']}")
        print()


for rule_id in RULES:
    show_category_examples(f"{rule_id}_lost", diagnosed_categories[f"{rule_id}_lost"])
    show_category_examples(f"{rule_id}_gained", diagnosed_categories[f"{rule_id}_gained"])

R1_lost: 4 examples (1.94% of rule disagreements)
Example 1
  sent_id: dev-s1065
  sentence_text: शुक्रवार की रात को जब एक आतंकी के मारे जाने के बाद दूसरा आतंकी भी कुछ देर फायरिंग करने के बाद इमारत के एक कमरे में छिपकर बैठ गया तो सुरक्षा - बलों व अन्य सभी एजेंसियों ने मान लिया कि वह भी मारा गया है ।
  token_form: बलों
  gold_deprel: nsubj
  stanza_deprel: nsubj
  gold_case_marker: ने
  stanza_case_marker: 
  gold_rule: R1
  stanza_rule: 
  gold_final_decision: confirmed
  stanza_final_decision: mapping_hypothesis
  diagnosis: case marker changed ने to empty

Example 2
  sent_id: dev-s194
  sentence_text: दूरसंचार सेवा मुहैया कराने वाली टाटा टेलिसर्विस लिमिटेड और रिलायंस इंफोकॉम ने बाजार में कुछ समय पहले फिक्सड वायरलेस फोन उतारे थे ।
  token_form: लिमिटेड
  gold_deprel: nsubj
  stanza_deprel: nsubj
  gold_case_marker: ने
  stanza_case_marker: 
  gold_rule: R1
  stanza_rule: 
  gold_final_decision: confirmed
  stanza_final_decision: mapping_hypothesis
  diagnosis: case marker changed ने 

## 6. Save Output CSV Files

In [6]:
def write_csv(filepath, rows, fieldnames=None):
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else EXAMPLE_FIELDS + ["diagnosis"]
    with open(filepath, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


SUMMARY_FIELDS = [
    "category",
    "count",
    "percent_of_rule_disagreements",
    "most_common_cause",
]
EXAMPLE_OUTPUT_FIELDS = EXAMPLE_FIELDS + ["diagnosis"]

write_csv(RESULTS_DIR / "rule_disagreement_summary.csv", summary_rows, SUMMARY_FIELDS)

for rule_id in RULES:
    lost_name = f"{rule_id}_lost"
    gained_name = f"{rule_id}_gained"
    write_csv(
        RESULTS_DIR / f"{rule_id}_lost_examples.csv",
        diagnosed_categories[lost_name],
        EXAMPLE_OUTPUT_FIELDS,
    )
    write_csv(
        RESULTS_DIR / f"{rule_id}_gained_examples.csv",
        diagnosed_categories[gained_name],
        EXAMPLE_OUTPUT_FIELDS,
    )

print("Saved outputs:")
print("  results/rule_disagreement_summary.csv")
for rule_id in RULES:
    print(f"  results/{rule_id}_lost_examples.csv")
    print(f"  results/{rule_id}_gained_examples.csv")

Saved outputs:
  results/rule_disagreement_summary.csv
  results/R1_lost_examples.csv
  results/R1_gained_examples.csv
  results/R2_lost_examples.csv
  results/R2_gained_examples.csv
  results/R3_lost_examples.csv
  results/R3_gained_examples.csv
  results/R4_lost_examples.csv
  results/R4_gained_examples.csv
  results/R5_lost_examples.csv
  results/R5_gained_examples.csv


## 7. Diagnostic Notes

- Lost rule categories show where gold UD triggered a verifier rule but Stanza did not.
- Gained rule categories show where Stanza triggered a verifier rule but gold UD did not.
- The diagnosis column reports the direct visible difference: deprel change, case-marker change, or both.
- These outputs are for parser error analysis only. They do not propose or implement fixes.